# 黑马头条项目｜第04天：自媒体文章审核

> 目标：在文章提交后自动审核文本和图片，审核通过后通过 Feign 保存 APP 端文章；补充服务降级、异步执行、自管理敏感词、OCR 图片文字审核和文章详情静态化。

本篇根据百度网盘第04天“自媒体文章审核”讲义整理。源码只作静态参照，未启动阿里云、Nacos、MySQL、MinIO、微服务或 Nginx，未调用第三方审核接口。讲义中的 AccessKey、固定 IP、账号、密码和本地路径均已替换为占位符。

## 1. 审核业务总流程

```text
自媒体提交文章（wm_news.status = 1）
              │
              ▼
提取正文文本 + 内容图片 + 封面图片
              │
              ├─ 自管理敏感词过滤
              ├─ 第三方文本内容安全
              └─ 下载 MinIO 图片 → OCR/图片内容安全
                         │
          block ─────────┴──────── review
           │                              │
   status = 2 审核失败             status = 3 人工审核
           │
           └────全部通过────▶ Feign 保存 APP 文章三张表
                                  │
                         回填 article_id
                                  │
                         status = 9 已发布
```

审核必须以 `wm_news.status = 1`（待审核）为入口。任何失败都要保存原因，审核成功后才创建 APP 端数据。

## 2. 内容安全服务

内容安全服务可以检测文本、图片、视频、语音等内容。讲义选择阿里云内容安全，项目使用文本和图片审核接口。实际接入时需要在安全的配置中心保存密钥，并限制权限、轮换周期和调用额度。

### 2.1 接入边界

- 文本审核输入：文章标题与正文纯文本拼接；
- 图片审核输入：从文章内容和封面提取 URL，从 MinIO 下载成字节数组；
- 第三方返回建议通常分为通过、`block`（拦截）和 `review`（转人工）；
- 调用超时、网络失败和返回格式异常要进入可追踪的失败处理，不应把异常文章误标为已发布。

示例配置使用占位符：

```yaml
aliyun:
  accessKeyId: <ALIYUN_ACCESS_KEY_ID>
  secret: <ALIYUN_ACCESS_KEY_SECRET>
  scenes: terrorism
```

本地测试可以注入 `GreenTextScan`、`GreenImageScan`，但本笔记没有执行任何真实审核调用。

## 3. APP 端文章保存接口

审核通过后，需要在文章微服务保存：

- `ap_article`：文章标题、作者、频道、布局、封面和发布时间；
- `ap_article_config`：评论、转发、下架和删除开关；
- `ap_article_content`：正文 JSON。

### 3.1 Feign 接口

```java
@Data
public class ArticleDto extends ApArticle {
    private String content;
}

@FeignClient(value = "leadnews-article",
             fallback = IArticleClientFallback.class)
public interface IArticleClient {
    @PostMapping("/api/v1/article/save")
    ResponseResult saveArticle(@RequestBody ArticleDto dto);
}
```

文章服务实现同名接口，调用 `ApArticleService.saveArticle`。新增时保存三张表，修改时更新文章和内容；配置对象默认允许评论、转发，初始未删除、未下架。三张表需要保持同一事务边界。

## 4. 分布式 ID：Snowflake

文章数据后续可能进行分库分表，自增 ID 在不同数据库中容易冲突，因此文章端使用雪花算法生成 Long 型 ID。

Snowflake 的典型位分配是：符号位 1 位、毫秒时间 41 位、机器标识 10 位、毫秒内序列 12 位。机器标识要在部署环境中保证唯一，时钟回拨也要有防护策略。

MyBatis-Plus 示例：

```java
@TableId(value = "id", type = IdType.ID_WORKER)
private Long id;
```

```yaml
mybatis-plus:
  global-config:
    datacenter-id: <DATACENTER_ID>  # 0-31
    workerId: <WORKER_ID>            # 0-31
```

`datacenter-id` 和 `workerId` 不能在多实例部署中重复。

## 5. 自动审核服务实现

### 5.1 审核主方法

`WmNewsAutoScanService` 暴露 `autoScanWmNews(Integer id)`。实现类按顺序完成：

1. 查询 `wm_news`，不存在则报业务异常；
2. 只处理待审核状态；
3. 从内容 JSON 提取纯文本和图片，并合并封面图片；
4. 执行自管理敏感词检查；
5. 调用第三方文本审核；
6. 从 MinIO 下载图片并执行图片文字/OCR和图片内容审核；
7. 全部通过后组装 `ArticleDto` 调用 Feign；
8. 回填 `article_id`，把 `wm_news.status` 更新为 9。

### 5.2 文本与图片提取

```java
List<Map> nodes = JSONArray.parseArray(wmNews.getContent(), Map.class);
StringBuilder text = new StringBuilder();
List<String> images = new ArrayList<>();
for (Map node : nodes) {
    if ("text".equals(node.get("type"))) text.append(node.get("value"));
    if ("image".equals(node.get("type"))) images.add((String) node.get("value"));
}
if (StringUtils.isNotBlank(wmNews.getImages())) {
    images.addAll(Arrays.asList(wmNews.getImages().split(",")));
}
```

图片 URL 要去重，避免同一对象重复下载和重复审核。

## 6. 审核结果与状态流转

| 状态 | 含义 | 典型触发 |
|---:|---|---|
| 0 | 草稿 | 尚未提交审核 |
| 1 | 待审核 | 自媒体提交 |
| 2 | 审核失败 | 第三方 `block` 或命中自管理敏感词 |
| 3 | 人工审核 | 第三方返回 `review` 或需要人工判断 |
| 4 | 人工审核通过 | 人工确认通过 |
| 8 | 审核通过，待发布 | 等待发布调度 |
| 9 | 已发布 | APP 文章已保存并回填 ID |

统一更新方法应同时写入 `status` 与 `reason`：

```java
private void updateWmNews(WmNews news, short status, String reason) {
    news.setStatus(status);
    news.setReason(reason);
    wmNewsMapper.updateById(news);
}
```

异常、超时和第三方服务不可用时，需要区分“审核失败”“人工复核”和“系统重试”，不能把所有异常都标记为违规。

## 7. 文本审核与图片审核

### 7.1 文本审核

将标题和正文拼接后调用 `GreenTextScan`。返回 `block` 时更新为 2，返回 `review` 时更新为 3；审核方法返回 `false`，主流程立即停止。

### 7.2 图片审核

1. 对图片 URL 去重；
2. 用 `FileStorageService.downLoadFile` 从 MinIO 获取字节；
3. 将字节数组组成图片审核请求；
4. `block` 更新为 2，`review` 更新为 3；
5. 图片下载失败要记录日志并进入重试/人工路径。

外部服务调用要设置超时、限流和重试上限，避免审核线程长期阻塞。

## 8. Feign 服务降级

服务降级用于保护下游文章服务：当文章服务超时或不可用时，自媒体服务快速返回失败结果，不让请求线程一直阻塞。

```java
@Component
public class IArticleClientFallback implements IArticleClient {
    @Override
    public ResponseResult saveArticle(ArticleDto dto) {
        return ResponseResult.errorResult(
            AppHttpCodeEnum.SERVER_ERROR, "获取数据失败");
    }
}
```

Feign 客户端指向降级类：

```java
@FeignClient(value = "leadnews-article",
             fallback = IArticleClientFallback.class)
```

并在客户端配置中开启降级、连接超时和读取超时。降级结果要进入监控和补偿队列，避免“返回失败”后丢失待审核文章。

## 9. 异步审核

文章提交接口不必等待文本、图片和跨服务保存全部完成。提交成功后调用审核服务，审核方法使用 `@Async`，启动类使用 `@EnableAsync`。

```java
@Async
@Override
public void autoScanWmNews(Integer id) {
    // 查询文章、审核、保存 APP 文章、回写状态
}
```

```java
@SpringBootApplication
@EnableFeignClients(basePackages = "com.heima.apis")
@EnableAsync
public class WemediaApplication { }
```

异步边界需要注意：

- 事务提交后再触发审核，避免异步线程读不到刚写入的数据；
- ThreadLocal、请求上下文和登录信息不会自动传入异步线程；审核只使用文章 ID重新查询；
- 线程池要有容量、队列、拒绝策略和监控；
- 失败要可重试、可查询，不能只打印堆栈。

## 10. 自管理敏感词与 DFA

第三方接口无法覆盖企业自定义词库，因此新增 `wm_sensitive` 表，字段包括敏感词和创建时间。审核时加载词库并调用 `SensitiveWordUtil`。

DFA（Deterministic Finite Automaton，确定有穷自动机）把敏感词拆成多级 Map 节点：共享前缀，匹配时沿字符逐级查找。相较于逐词 `indexOf`，大量词条下能减少重复扫描。

审核流程：

1. 查询敏感词集合；
2. 初始化 DFA 词典；
3. 在标题和正文文本中匹配；
4. 找到结果就写入状态 2 和命中原因；
5. 没有命中才继续第三方文本审核。

词库加载可以做缓存并在变更时刷新，避免每篇文章都从数据库重建整棵 DFA。命中结果不应把完整敏感词泄露到普通日志。

## 11. OCR 图片文字审核

图片可能包含文字，单纯的图片内容审核无法覆盖自管理敏感词。OCR（Optical Character Recognition）先把图片转成文本，再复用敏感词匹配。

讲义使用 Tess4J 封装 Tesseract：

```xml
<dependency>
  <groupId>net.sourceforge.tess4j</groupId>
  <artifactId>tess4j</artifactId>
  <version>4.1.1</version>
</dependency>
```

```java
@Component
@ConfigurationProperties(prefix = "tess4j")
public class Tess4jClient {
    private String dataPath;
    private String language;

    public String doOCR(BufferedImage image) throws TesseractException {
        ITesseract tesseract = new Tesseract();
        tesseract.setDatapath(dataPath);
        tesseract.setLanguage(language);
        return tesseract.doOCR(image)
            .replaceAll("\r|\n", "-")
            .replaceAll(" ", "");
    }
}
```

图片审核顺序可以是：下载 → OCR → 自管理敏感词 → 组装图片字节 → 第三方图片审核。OCR 识别质量受字体、分辨率和语言包影响，识别异常要进入人工或重试路径。

## 12. 文章详情静态页生成

APP 文章保存成功后，文章微服务异步生成详情 HTML：

1. 使用 `article.ftl` 和正文 JSON 渲染 HTML；
2. 调用 `FileStorageService.uploadHtmlFile` 上传 MinIO；
3. 将返回地址回写 `ap_article.static_url`。

```java
@Async
public void buildArticleToMinIO(ApArticle article, String content) {
    if (StringUtils.isBlank(content)) return;
    Template template = configuration.getTemplate("article.ftl");
    StringWriter out = new StringWriter();
    Map<String, Object> model = new HashMap<>();
    model.put("content", JSONArray.parseArray(content));
    template.process(model, out);

    try (InputStream in = new ByteArrayInputStream(
            out.toString().getBytes(StandardCharsets.UTF_8))) {
        String url = fileStorageService.uploadHtmlFile(
            "", article.getId() + ".html", in);
        articleService.updateStaticUrl(article.getId(), url);
    }
}
```

HTML 上传成功与数据库回写之间可能出现不一致，应设计幂等 Key、重试、旧对象清理和状态监控。

## 13. 综合测试清单

服务启动顺序：Nacos → article 微服务 → wemedia 微服务 → wemedia 网关 → wemedia 前端。资料中的测试场景：

1. 正常文章：审核成功，APP 文章三张表有记录，`wm_news.article_id` 回填，状态变为 9；
2. 敏感文本：状态变为 2，原因保存；
3. 敏感图片：状态变为 2，原因保存；
4. 不确定内容：状态变为 3，进入人工审核；
5. article 服务超时：Feign 触发降级，文章保持可追踪状态；
6. 静态化成功：`static_url` 回写，HTML 对象可访问。

本次只完成资料整理和静态校验，没有执行上述环境和接口测试。

## 14. 复习与工程注意事项

- 自动审核是状态机：待审核、失败、人工、通过和已发布之间要有明确迁移规则。
- 审核输入必须从数据库和对象存储读取，不能信任客户端直接传来的用户身份或素材归属。
- 自管理敏感词优先于第三方审核，词库应缓存、可更新、可审计。
- OCR 是额外的 CPU/内存和耗时步骤，应限流、超时并避免重复识别。
- Feign 降级只能快速失败，仍需要重试或补偿机制保证最终一致性。
- 异步审核和静态化要有任务状态、日志、监控和幂等设计。
- 密钥只从密钥管理或环境变量注入，禁止提交到代码、配置仓库或日志。

### 自测题

1. 为什么审核成功后才创建 APP 端文章？
2. `block` 与 `review` 分别应该进入哪些状态？
3. 文章服务保存成功但静态 HTML 上传失败时，如何补偿？
4. 为什么异步审核不能依赖提交请求中的 ThreadLocal？
5. DFA 和 `String.indexOf` 在大量敏感词场景下有什么差别？